In [1]:
import celloracle as co

oracle = co.load_hdf5("/home/elicanter/cell_oracle_run/celloracle_hematopoiesis_imputed.celloracle.oracle")

print(oracle.adata.obs.columns)

/home/elicanter/miniconda3/envs/celloracle_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Index(['plate', 'well', 'facs_fsc.area', 'facs_fsc_height', 'facs_ssc.area',
       'facs_ssc_height', 'facs_cd45ra', 'facs_cd135', 'facs_cd90',
       'facs_cd49f', 'facs_lin', 'facs_cd34', 'facs_cd38', 'facs_cd10',
       'culture_key', 'culture_cells_in_colony', 'culture_mk_cd41+',
       'culture_ery_cd235+', 'culture_total_number_of_my_cd33+',
       'facs_pseudotime', 'fate_stemness', 'n_genes', 'dpt_pseudotime',
       'leiden'],
      dtype='object')


In [5]:
import numpy as np

clusters = oracle.adata.obs["leiden"]

print(clusters)

I1_plate10_A_10    2
I1_plate10_A_11    3
I1_plate10_A_12    5
I1_plate10_A_1     0
I1_plate10_A_4     2
                  ..
I1_plate9_H_12     5
I1_plate9_H_1      2
I1_plate9_H_2      4
I1_plate9_H_5      2
I1_plate9_H_9      1
Name: leiden, Length: 1035, dtype: category
Categories (9, object): ['0', '1', '2', '3', ..., '5', '6', '7', '8']


In [6]:
import numpy as np
from scipy.stats import spearmanr
from tqdm import tqdm

BASE = "/home/elicanter/bert_grn_out_61_masking2"
N_RUNS = 5

# -----------------------------
# Load CellOracle global
# -----------------------------
co_A = np.load("/home/elicanter/cell_oracle_run/cell_oracle_global_A/A_celloracle.npy")
co_genes = np.load("/home/elicanter/cell_oracle_run/cell_oracle_global_A/A_genes.npy", allow_pickle=True).astype(str)
co_tfs   = np.load("/home/elicanter/cell_oracle_run/cell_oracle_global_A/A_tfs.npy", allow_pickle=True).astype(str)

if co_A.shape != (len(co_genes), len(co_tfs)):
    co_A = co_A.T

# -----------------------------
# Load YOUR gene/TF universe (from run0)
# -----------------------------
A_genes = np.load(f"{BASE}/run0/A_genes.npy", allow_pickle=True).astype(str)
A_tfs   = np.load(f"{BASE}/run0/A_tfs.npy", allow_pickle=True).astype(str)

common_genes = np.array(sorted(set(A_genes) & set(co_genes)))
common_tfs   = np.array(sorted(set(A_tfs) & set(co_tfs)))

print("Shared genes:", len(common_genes), "Shared TFs:", len(common_tfs))

Shared genes: 2995 Shared TFs: 277


In [7]:
g_idx_pred = np.array([list(A_genes).index(g) for g in common_genes])
t_idx_pred = np.array([list(A_tfs).index(t) for t in common_tfs])

g_idx_co = np.array([list(co_genes).index(g) for g in common_genes])
t_idx_co = np.array([list(co_tfs).index(t) for t in common_tfs])

co_A = co_A[np.ix_(g_idx_co, t_idx_co)]

In [9]:
co_cluster_A = []
for k in range(9):
    A_k = np.load(f"/home/elicanter/cell_oracle_run/A_clusters/A_cluster_{k}.npy")

    if A_k.shape != (len(co_genes), len(co_tfs)):
        A_k = A_k.T

    A_k = A_k[np.ix_(g_idx_co, t_idx_co)]
    co_cluster_A.append(A_k)

co_cluster_A = np.stack(co_cluster_A)   # (K=9, G, T)

In [10]:
import anndata as ad

# assuming oracle.adata exists in memory
clusters = oracle.adata.obs["leiden"].astype(int).values

In [11]:
global_hub_rhos = []
global_tf_rhos = []
global_flat_rhos = []

for r in range(N_RUNS):
    weights = np.load(f"{BASE}/run{r}/mix_weights.npy")   # (cells, D)
    A_dict  = np.load(f"{BASE}/run{r}/A_dict.npy")        # (D, G, T)

    # mean GRN
    w_mean = weights.mean(axis=0)
    A_pred = np.tensordot(w_mean, A_dict, axes=(0, 0))    # (G, T)

    # align
    A_pred = A_pred[np.ix_(g_idx_pred, t_idx_pred)]

    # flatten Spearman
    flat_rho, _ = spearmanr(A_pred.ravel(), co_A.ravel())
    global_flat_rhos.append(flat_rho)

    # hub
    hub_pred = np.abs(A_pred).sum(axis=0)
    hub_co   = np.abs(co_A).sum(axis=0)
    hub_rho, _ = spearmanr(hub_pred, hub_co)
    global_hub_rhos.append(hub_rho)

    # per-TF
    tf_rhos = []
    for ti in range(A_pred.shape[1]):
        x = np.abs(A_pred[:, ti])
        y = np.abs(co_A[:, ti])
        if x.std() < 1e-8 or y.std() < 1e-8:
            continue
        tf_rhos.append(spearmanr(x, y)[0])

    global_tf_rhos.append(np.mean(tf_rhos))

print("\nGLOBAL vs CellOracle")
print("Flat rho:", np.mean(global_flat_rhos), "+/-", np.std(global_flat_rhos))
print("Hub rho :", np.mean(global_hub_rhos), "+/-", np.std(global_hub_rhos))
print("TF rho  :", np.mean(global_tf_rhos), "+/-", np.std(global_tf_rhos))


GLOBAL vs CellOracle
Flat rho: 0.2049470072365481 +/- 0.0017368618439081455
Hub rho : 0.8800939283104986 +/- 0.002947753120355545
TF rho  : 0.9514127066210823 +/- 0.0005609654751960617


In [12]:
cell_vs_global = []
cell_vs_cluster = []

for r in range(N_RUNS):
    weights = np.load(f"{BASE}/run{r}/mix_weights.npy")   # (cells, D)
    A_dict  = np.load(f"{BASE}/run{r}/A_dict.npy")

    for i in tqdm(range(weights.shape[0])):

        w = weights[i]
        A_cell = np.tensordot(w, A_dict, axes=(0, 0))  # (G, T)
        A_cell = A_cell[np.ix_(g_idx_pred, t_idx_pred)]

        # flatten global comparison
        cell_vs_global.append(
            spearmanr(A_cell.ravel(), co_A.ravel())[0]
        )

        # cluster-specific comparison
        cl = clusters[i]
        A_cl = co_cluster_A[cl]

        cell_vs_cluster.append(
            spearmanr(A_cell.ravel(), A_cl.ravel())[0]
        )

 33%|███▎      | 345/1035 [02:43<05:27,  2.10it/s]


KeyboardInterrupt: 

In [14]:
#faster
import numpy as np
from scipy.stats import rankdata
from tqdm import tqdm

cell_vs_global = []
cell_vs_cluster = []

# -----------------------------
# PRECOMPUTE GLOBALS ONCE
# -----------------------------
co_A_flat = co_A.ravel()

co_A_rank = rankdata(co_A.ravel())

co_cluster_A_rank = [
    rankdata(co_cluster_A[k].ravel())
    for k in range(co_cluster_A.shape[0])
]

g_idx_pred = np.array(g_idx_pred)
t_idx_pred = np.array(t_idx_pred)

# -----------------------------
# MAIN LOOP
# -----------------------------
for r in range(N_RUNS):
    weights = np.load(f"{BASE}/run{r}/mix_weights.npy")   # (cells, D)
    A_dict  = np.load(f"{BASE}/run{r}/A_dict.npy")        # (D, G, T)

    D = weights.shape[1]

    for i in tqdm(range(weights.shape[0])):

        w = weights[i]

        # fast linear combination (same as tensordot but slightly faster/clearer)
        A_cell = np.einsum("d,dgt->gt", w, A_dict)

        # subset once
        A_cell = A_cell[np.ix_(g_idx_pred, t_idx_pred)]

        # flatten ONCE
        A_flat = A_cell.ravel()

        # rank ONCE
        A_rank = rankdata(A_flat)

        # -------------------------
        # GLOBAL SPEARMAN (via Pearson on ranks)
        # -------------------------
        cell_vs_global.append(
            np.corrcoef(A_rank, co_A_rank)[0, 1]
        )

        # -------------------------
        # CLUSTER SPEARMAN
        # -------------------------
        cl = clusters[i]
        cell_vs_cluster.append(
            np.corrcoef(A_rank, co_cluster_A_rank[cl])[0, 1]
        )

100%|██████████| 1035/1035 [02:12<00:00,  7.78it/s]


In [15]:
print("\nCELL-LEVEL GRN MATCH")
print("Global CellOracle match:",
      np.mean(cell_vs_global), "+/-", np.std(cell_vs_global))

print("Cluster CellOracle match:",
      np.mean(cell_vs_cluster), "+/-", np.std(cell_vs_cluster))


CELL-LEVEL GRN MATCH
Global CellOracle match: 0.18689407712152176 +/- 0.019521644185921
Cluster CellOracle match: 0.09540739290093933 +/- 0.012658877842960546


In [16]:
cluster_scores_global = []
cluster_scores_local  = []

for k in range(9):

    idx = np.where(clusters == k)[0]

    # aggregate YOUR GRNs for cluster
    # (streaming average, no storage)
    A_sum = None
    count = 0

    for r in range(N_RUNS):
        weights = np.load(f"{BASE}/run{r}/mix_weights.npy")
        A_dict  = np.load(f"{BASE}/run{r}/A_dict.npy")

        A_cluster = np.zeros((len(common_genes), len(common_tfs)))

        for i in idx:
            A_i = np.tensordot(weights[i], A_dict, axes=(0, 0))
            A_i = A_i[np.ix_(g_idx_pred, t_idx_pred)]
            A_cluster += A_i
            count += 1

        A_cluster /= len(idx)

        # compare to CellOracle cluster GRN
        cluster_scores_global.append(
            spearmanr(A_cluster.ravel(), co_A.ravel())[0]
        )

        cluster_scores_local.append(
            spearmanr(A_cluster.ravel(), co_cluster_A[k].ravel())[0]
        )

print("\nCLUSTER LEVEL")
print("vs global CO:", np.mean(cluster_scores_global))
print("vs cluster CO:", np.mean(cluster_scores_local))


CLUSTER LEVEL
vs global CO: 0.18497762680453797
vs cluster CO: 0.09444809305142865


In [18]:
cell_rhos = []

for i in tqdm(range(weights.shape[0])):
    cell_grns = []

    for r in range(N_RUNS):
        weights = np.load(f"{BASE}/run{r}/mix_weights.npy")
        A_dict  = np.load(f"{BASE}/run{r}/A_dict.npy")

        A_i = np.tensordot(weights[i], A_dict, axes=(0,0))
        A_i = A_i[np.ix_(g_idx_pred, t_idx_pred)]
        cell_grns.append(A_i.ravel())

    # compare all pairs for this cell
    rhos = []
    for a in range(N_RUNS):
        for b in range(a+1, N_RUNS):
            rhos.append(spearmanr(cell_grns[a], cell_grns[b])[0])

    cell_rhos.append(np.mean(rhos))

cell_rhos = np.array(cell_rhos)

print("Cell-level GRN reproducibility:")
print("mean:", cell_rhos.mean())
print("std :", cell_rhos.std())

  0%|          | 3/1035 [00:08<46:19,  2.69s/it]


KeyboardInterrupt: 

In [21]:
spearmanr(
    vec(global_celloracle),
    vec(cluster_celloracle)
)

NameError: name 'vec' is not defined